# Phase 6.8 — volume feature comparison

This notebook tests one feature: `volume_ratio_20d`. It means today’s company trading volume divided by the company’s average volume over the latest 20 trading days. It compares all 14 approved market inputs with the same set **without this volume feature**.

Important: this project’s historical volume comes from Alpaca’s IEX feed. It is useful for this research comparison, but it is not complete whole-market volume.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'src' / 'model_training').is_dir():
    root = root.parent
if not (root / 'src' / 'model_training').is_dir():
    raise FileNotFoundError('Open this notebook from inside the HalalifyAI repository.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
%cd {root}
print(f'Project folder: {root}')

## Run the controlled comparison

The next cell uses only the verified Phase 6.2 recovery dataset and its time-based protocol. It tunes only on earlier inner-validation data, then scores each later outer fold once. Expect a few minutes of work.

In [ ]:
from src.model_training.feature_ablation.ablation_runner import (
    DEFAULT_RECOVERY_VOLUME_OUTPUT,
    write_recovery_volume_ablation_report,
)

report = write_recovery_volume_ablation_report()
print('Status:', report['status'])
print('Volume limitation:', report['volume_source_limitation'])
print('Evidence folder:', DEFAULT_RECOVERY_VOLUME_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means the trailing volume ratio consistently helped. `reject` means it did not help in most periods. `inconclusive` means the evidence is mixed. For risk, a result cannot be retained if High-risk detection becomes worse or more High-risk outcomes are wrongly predicted as Low risk.

In [ ]:
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])